# FIFA World Cup 2026 — Notebook 07: Elo Model (Fix 1)

## About

**Purpose:** Replace the simple goal-ratio strengths with an **Elo rating** that is opponent-adjusted and includes a home-advantage term — then prove (or disprove) that it beats the Poisson baseline using notebook 06's metrics.<br>
**Author:** Ganapathy K<br>
**Date:** 2026-06-07<br>
**Notes:** Elo updates a team's rating after every match by *how surprising* the result was against *that specific opponent* — so beating a strong side gains more than beating a weak one (the opponent-adjustment that the ratio model lacked, which over-rated Morocco). A home-advantage term in Elo points addresses the under-prediction of home wins that nb 06's calibration curve exposed. The single Elo number is mapped to expected goals (via a goal-supremacy fit on the training data) so the existing Poisson grid → W/D/L machinery is reused unchanged. Scored on the **same** time-split test set as nb 06 for an apples-to-apples comparison.<br>
**Description:** Reads `played_matches.parquet`; compares against `eval_metrics.parquet` from notebook 06.

### Change Control

| Date       | Version | Author      | Changes         |
|------------|---------|-------------|-----------------|
| 2026-06-07 | 1.0     | Ganapathy K | Initial version |


## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

### 1.1 Imports

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import poisson
from pathlib import Path

### 1.2 Config

Elo knobs: every team starts at `INITIAL_RATING`; `K_FACTOR` is how fast ratings move per match; `HOME_ADVANTAGE_ELO` is the rating bonus a non-neutral home side gets. `CUTOFF_DATE` and `MIN_TRAIN_GAMES` match notebook 06 exactly so the test set is identical.

In [3]:
PROCESSED_DATA_DIR = Path(r"D:/Data Science/Visual Studio Code/fifa_wc_2026_poisson/data/processed")
PLAYED_MATCHES_PATH = PROCESSED_DATA_DIR / "played_matches.parquet"
BASELINE_METRICS_PATH = PROCESSED_DATA_DIR / "eval_metrics.parquet"
ELO_METRICS_PATH = PROCESSED_DATA_DIR / "elo_metrics.parquet"
ELO_RATINGS_PATH = PROCESSED_DATA_DIR / "elo_ratings.parquet"

CUTOFF_DATE = pd.Timestamp("2023-01-01")
MIN_TRAIN_GAMES = 10
MAX_GOALS = 10

INITIAL_RATING = 1500.0
K_FACTOR = 30.0
HOME_ADVANTAGE_ELO = 65.0

## 2. Train/Test Split

Same split as notebook 06: train on matches before the cutoff, hold out matches after. Elo will be built walking the training matches in date order.

In [4]:
played_matches = pd.read_parquet(PLAYED_MATCHES_PATH).sort_values("date").reset_index(drop=True)

train_matches = played_matches[played_matches["date"] < CUTOFF_DATE]
test_matches = played_matches[played_matches["date"] >= CUTOFF_DATE].copy()
print(f"Train matches: {len(train_matches)} | Test matches: {len(test_matches)}")

Train matches: 45806 | Test matches: 3500


## 3. Build Elo Ratings

Walk the training matches in chronological order. For each: the expected home score comes from the rating gap (plus home bonus if not neutral); the actual score is 1 / 0.5 / 0 for win / draw / loss. The rating moves by `K × (actual − expected)`, scaled up for bigger winning margins (a 4-0 shifts ratings more than a 1-0). We also log each match's pre-game rating gap and goal difference — needed in section 4 to map Elo onto goals.

In [5]:
def margin_multiplier(goal_difference):
    margin = abs(goal_difference)
    if margin <= 1:
        return 1.0
    if margin == 2:
        return 1.5
    return (11 + margin) / 8   # 3-goal win -> 1.75, grows slowly after


ratings = {}
gap_history = []   # pre-match (home_rating + home_adv - away_rating)
goal_diff_history = []

for match in train_matches.itertuples(index=False):
    home_rating = ratings.get(match.home_team, INITIAL_RATING)
    away_rating = ratings.get(match.away_team, INITIAL_RATING)
    home_bonus = 0.0 if match.neutral else HOME_ADVANTAGE_ELO

    rating_gap = home_rating + home_bonus - away_rating
    expected_home = 1.0 / (1.0 + 10.0 ** (-rating_gap / 400.0))

    goal_diff = match.home_score - match.away_score
    if goal_diff > 0:
        actual_home = 1.0
    elif goal_diff == 0:
        actual_home = 0.5
    else:
        actual_home = 0.0

    gap_history.append(rating_gap)
    goal_diff_history.append(goal_diff)

    change = K_FACTOR * margin_multiplier(goal_diff) * (actual_home - expected_home)
    ratings[match.home_team] = home_rating + change
    ratings[match.away_team] = away_rating - change

# count training appearances per team (for the same filter as nb 06)
appearances = pd.concat([train_matches["home_team"], train_matches["away_team"]]).value_counts()
print(f"Rated {len(ratings)} teams.")
top = sorted(ratings.items(), key=lambda kv: kv[1], reverse=True)[:10]
for team, rating in top:
    print(f"  {team:<15} {rating:6.0f}")

Rated 333 teams.
  Brazil            2191
  Argentina         2154
  Netherlands       2086
  France            2076
  Spain             2062
  Portugal          2037
  England           2014
  Germany           2005
  Belgium           1999
  Italy             1995


## 4. Map Elo to Expected Goals

Elo gives one number per team; the Poisson grid needs *expected goals for each side*. Bridge them with a fit on the training data: regress actual goal difference on the pre-match rating gap to learn how many goals a rating gap is worth (`supremacy`). Split that around the average total goals to get each side's expected goals, then the existing Poisson grid produces W/D/L.

In [6]:
slope, intercept = np.polyfit(gap_history, goal_diff_history, 1)
total_goals = (train_matches["home_score"] + train_matches["away_score"]).mean()
print(f"Supremacy fit: goal_diff = {slope:.5f} * rating_gap + {intercept:.3f}")
print(f"Average total goals per match: {total_goals:.3f}")


def predict_wdl(home_team, away_team, neutral):
    home_rating = ratings.get(home_team, INITIAL_RATING)
    away_rating = ratings.get(away_team, INITIAL_RATING)
    home_bonus = 0.0 if neutral else HOME_ADVANTAGE_ELO
    rating_gap = home_rating + home_bonus - away_rating

    supremacy = slope * rating_gap + intercept
    expected_home = max((total_goals + supremacy) / 2.0, 0.05)
    expected_away = max((total_goals - supremacy) / 2.0, 0.05)

    goals = np.arange(0, MAX_GOALS + 1)
    grid = np.outer(poisson.pmf(goals, expected_home), poisson.pmf(goals, expected_away))
    p_home = np.tril(grid, -1).sum()
    p_draw = np.trace(grid)
    p_away = np.triu(grid, 1).sum()
    total = p_home + p_draw + p_away
    return p_home / total, p_draw / total, p_away / total

Supremacy fit: goal_diff = 0.00587 * rating_gap + 0.243
Average total goals per match: 2.952


## 5. Predict & Score the Held-out Matches

Identical filter to notebook 06 (both teams need at least `MIN_TRAIN_GAMES` training games) so the test set matches exactly. Outcome coding: 0 home win, 1 draw, 2 away win.

In [7]:
rated_enough = set(appearances[appearances >= MIN_TRAIN_GAMES].index)
mask = test_matches["home_team"].isin(rated_enough) & test_matches["away_team"].isin(rated_enough)
eval_matches = test_matches[mask].copy()

predictions = [predict_wdl(h, a, n) for h, a, n
               in zip(eval_matches["home_team"], eval_matches["away_team"], eval_matches["neutral"])]
probs = np.array(predictions)

outcome = np.where(eval_matches["home_score"] > eval_matches["away_score"], 0,
                   np.where(eval_matches["home_score"] == eval_matches["away_score"], 1, 2))
actual_onehot = np.eye(3)[outcome]
print(f"Scoring {len(eval_matches)} test matches (same filter as nb 06)")

Scoring 3475 test matches (same filter as nb 06)


## 6. Metrics & Calibration

Same three metrics as nb 06 — Brier, log loss, RPS — plus the home-win calibration check. The calibration gap is the headline: did the home-advantage term close the under-prediction nb 06 found?

In [8]:
def ranked_probability_score(probs, outcomes):
    cum_pred = np.cumsum(probs, axis=1)
    cum_actual = np.cumsum(np.eye(3)[outcomes], axis=1)
    return np.mean(np.sum((cum_pred[:, :-1] - cum_actual[:, :-1]) ** 2, axis=1) / 2)


brier = np.mean(np.sum((probs - actual_onehot) ** 2, axis=1))
picked = probs[np.arange(len(probs)), outcome]
log_loss = -np.mean(np.log(np.clip(picked, 1e-15, 1)))
rps = ranked_probability_score(probs, outcome)
print(f"Elo  ->  Brier {brier:.4f} | Log loss {log_loss:.4f} | RPS {rps:.4f}")

bins = np.linspace(0, 1, 11)
calibration = pd.DataFrame({"p_home": probs[:, 0], "home_win": (outcome == 0).astype(int)})
calibration["bin"] = pd.cut(calibration["p_home"], bins, include_lowest=True)
calib_table = calibration.groupby("bin", observed=True).agg(
    matches=("home_win", "size"),
    mean_predicted_home=("p_home", "mean"),
    actual_home_rate=("home_win", "mean")).reset_index(drop=True)
calib_table["gap"] = (calib_table["actual_home_rate"] - calib_table["mean_predicted_home"]).round(3)
calib_table.round(3)

Elo  ->  Brier 0.5200 | Log loss 0.8839 | RPS 0.1727


,matches,mean_predicted_home,actual_home_rate,gap
0,275,0.044,0.058,0.014
1,273,0.150,0.147,-0.004
2,314,0.251,0.197,-0.053
3,321,0.352,0.312,-0.041
4,390,0.452,0.397,-0.055
5,388,0.549,0.459,-0.090
6,392,0.648,0.571,-0.076
7,374,0.748,0.631,-0.117
8,370,0.847,0.746,-0.101
9,378,0.943,0.915,-0.028


## 7. Head-to-Head vs the Poisson Baseline

Load notebook 06's metrics and put Elo beside them. Lower is better; this is the verdict on whether Fix 1 actually helped.

In [9]:
baseline_metrics = pd.read_parquet(BASELINE_METRICS_PATH)
comparison = baseline_metrics.copy()
comparison["Elo model"] = [brier, log_loss, rps]
comparison = comparison[["Base-rate benchmark", "Poisson model", "Elo model"]].round(4)
comparison["Elo vs Poisson %"] = ((comparison["Poisson model"] - comparison["Elo model"])
                                  / comparison["Poisson model"] * 100).round(1)

comparison.to_parquet(ELO_METRICS_PATH)
pd.Series(ratings, name="elo_rating").sort_values(ascending=False).to_frame().to_parquet(ELO_RATINGS_PATH)
print("Lower is better. Positive 'Elo vs Poisson %' = Elo improved on the baseline.\n")
print(comparison.to_string())

Lower is better. Positive 'Elo vs Poisson %' = Elo improved on the baseline.

          Base-rate benchmark  Poisson model  Elo model  Elo vs Poisson %
Brier                  0.6371         0.5703     0.5200               8.8
Log loss               1.0553         0.9625     0.8839               8.2
RPS                    0.2299         0.1978     0.1727              12.7
